# Lie Symmetry Analysis of the One-Dimensional Wave Equation

This tutorial demonstrates using **`symlie`** to analyze the linear hyperbolic wave equation:
$$u_{tt} - c^2 u_{xx} = 0$$

We investigate:
1. Variational formulation: Lagrangian density and the Euler–Lagrange equation.
2. The 8-dimensional symmetry space inside a total-degree-1 polynomial ansatz.
3. Commutator algebra and Lorentz invariance.
4. Symmetry reduction to D'Alembert's traveling wave solution $u(x, t) = f(x - ct) + g(x + ct)$.

In [ ]:
import sympy as sp

from symlie import (
    euler_lagrange,
    infinitesimals,
    lie_bracket,
    max_derivative_order,
    verify_generator,
)

sp.init_printing()

x, t = sp.symbols("x t")
c = sp.symbols("c", positive=True)
u = sp.Function("u")(x, t)

# Wave equation
wave_eq = u.diff(t, 2) - c**2 * u.diff(x, 2)
print("PDE Order:", max_derivative_order(wave_eq, u, (x, t)))
sp.Eq(wave_eq, 0)

## 1. Variational Formulation (Euler–Lagrange Operator)

The wave equation is the Euler–Lagrange equation for the Lagrangian density:
$$\mathcal{L} = \frac{1}{2} u_t^2 - \frac{c^2}{2} u_x^2$$

Applying `euler_lagrange` evaluates $\mathbf{E}_u(\mathcal{L}) = -u_{tt} + c^2 u_{xx}$:

In [ ]:
L = sp.Rational(1, 2) * u.diff(t) ** 2 - sp.Rational(1, 2) * c**2 * u.diff(x) ** 2
el_res = euler_lagrange(L, u, (x, t))

print("Euler-Lagrange Variational Derivative E_u(L):")
display(el_res[0])
assert sp.simplify(el_res[0] + wave_eq) == 0
print("Confirmed: E_u(L) = 0 reproduces the wave equation!")

## 2. Lie Point Symmetries

Solving with `infinitesimals(ansatz_degree=1)` computes an 8-dimensional polynomial-ansatz solution space (with $c=1$). This is not the complete point-symmetry algebra: the linear wave equation has the infinite solution-superposition ideal $h(x,t)\partial_u$, and in $1+1$ dimensions its characteristic-coordinate symmetries are also infinite-dimensional.

In [ ]:
sol = infinitesimals(wave_eq.subs(c, 1), u, (x, t), ansatz_degree=1)
print(f"Dimension within degree-1 ansatz: {sol.ansatz_dimension}\n")

for i, gen in enumerate(sol.basis, 1):
    is_valid = verify_generator(wave_eq.subs(c, 1), u, (x, t), gen)
    print(f"X_{i}: xi^x = {gen.xi[0]},  xi^t = {gen.xi[1]},  phi^u = {gen.phi[0]}")
    print(f"      Verified: {is_valid}")

## 3. Lorentz Invariance and Commutators

The **Lorentz boost generator** $X_{\text{Lorentz}} = t \partial_x + x \partial_t$ rotates space-time hyperbolically:
$$\tilde{x} = x \cosh(\varepsilon) + t \sinh(\varepsilon), \quad \tilde{t} = t \cosh(\varepsilon) + x \sinh(\varepsilon)$$

In [ ]:
from symlie import InfinitesimalGenerator

X_lorentz = InfinitesimalGenerator(xi=(t, x), phi=(0,))
X_trans_x = InfinitesimalGenerator(xi=(1, 0), phi=(0,))
X_trans_t = InfinitesimalGenerator(xi=(0, 1), phi=(0,))

print("[X_trans_x, X_lorentz] =", lie_bracket(X_trans_x, X_lorentz, u, (x, t)))
print("[X_trans_t, X_lorentz] =", lie_bracket(X_trans_t, X_lorentz, u, (x, t)))

## 4. D'Alembert's Traveling Wave Solution

Using translation invariance, any traveling wave profiles $f(x - ct)$ and $g(x + ct)$ solve the wave equation identically.

In [ ]:
f = sp.Function("f")(x - c * t)
g = sp.Function("g")(x + c * t)
dalembert = f + g

residual = sp.simplify(wave_eq.subs(u, dalembert).doit())
print("D'Alembert Solution Residual:", residual)
assert residual == 0
print("Verification: D'Alembert solution solves the wave equation identically!")